# Road Surface Classification — Training on Kaggle

Этот ноутбук демонстрирует обучение модели классификации дорожного покрытия на Kaggle с использованием:
- **DVC** для загрузки данных из Yandex Cloud
- **MLflow** для трекинга экспериментов
- **WandB** для визуализации

## Предварительные требования

1. Добавьте в Kaggle Secrets (Settings → Secrets):
   - `AWS_ACCESS_KEY_ID` — Yandex Cloud access key
   - `AWS_SECRET_ACCESS_KEY` — Yandex Cloud secret key
   - `BUCKET_NAME` — Имя бакета Yandex Cloud
   - `MLFLOW_TRACKING_URI` — URL вашего MLflow сервера
   - `WANDB_API_KEY` — Ключ WandB (опционально)

2. Подключите этот ноутбук к интернету (Settings → Internet → On)

3. Убедитесь, что включен GPU (Settings → Accelerator → GPU)

## 1. Настройка окружения

In [ ]:
# Установка зависимостей
!pip install -q torch torchaudio torchvision
!pip install -q mlflow wandb hydra-core omegaconf rich
!pip install -q librosa soundfile audiomentations
!pip install -q dvc dvc-s3
!pip install -q timm albumentations opencv-python

# Установка проекта (если есть в репозитории)
!pip install -q git+https://github.com/your-username/road-surface-classification.git

In [ ]:
# Импорт библиотек
import os
import sys
from pathlib import Path

import torch
import mlflow
import wandb

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 2. Настройка Secrets и DVC

In [ ]:
# Настройка переменных окружения из Kaggle Secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Yandex Cloud credentials для DVC
os.environ["AWS_ACCESS_KEY_ID"] = user_secrets.get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ["BUCKET_NAME"] = user_secrets.get_secret("BUCKET_NAME")

# MLflow tracking URI
os.environ["MLFLOW_TRACKING_URI"] = user_secrets.get_secret("MLFLOW_TRACKING_URI")

# WandB API key (опционально)
try:
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
    wandb_enabled = True
except Exception:
    wandb_enabled = False
    print("WANDB_API_KEY не найден, WandB логирование отключено")

print("✓ Переменные окружения настроены")

In [ ]:
# Инициализация DVC
from dvc.repo import Repo
from dvc.external_repo import external_repo

# Создаём временную директорию для клона репозитория
import tempfile
import shutil

WORK_DIR = Path(tempfile.mkdtemp())
REPO_URL = "https://github.com/your-username/road-surface-classification.git"

# Клонируем репозиторий
print(f"Клонирование репозитория в {WORK_DIR}...")
!git clone {REPO_URL} {WORK_DIR}

# Переходим в директорию репозитория
os.chdir(WORK_DIR)
print(f"✓ Репозиторий склонирован в {WORK_DIR}")

In [ ]:
# Настройка DVC remote
bucket_name = os.environ["BUCKET_NAME"]

!dvc init --no-scm
!dvc remote add -d storage s3://{bucket_name}/dvc-storage
!dvc remote modify storage endpointurl https://storage.yandexcloud.net

print("✓ DVC настроен")

In [ ]:
# Загрузка данных через DVC
print("Загрузка данных из Yandex Cloud...")
!dvc pull
print("✓ Данные загружены")

In [ ]:
# Проверка данных
data_dir = WORK_DIR / "data"
print(f"\nСтруктура данных:")
for item in data_dir.rglob("*"):
    if item.is_file():
        print(f"  {item.relative_to(data_dir)}")

## 3. Настройка MLflow

In [ ]:
# Импорт конфигурации MLflow из проекта
sys.path.insert(0, str(WORK_DIR / "src"))

from core.mlflow_config import setup_mlflow

# Настройка MLflow
mlflow_config = setup_mlflow(
    tracking_uri=os.environ["MLFLOW_TRACKING_URI"],
    experiment_name="road-surface-kaggle",
    artifact_location=f"s3://{os.environ['BUCKET_NAME']}/mlflow-artifacts"
)

print(f"✓ MLflow настроен: {mlflow.get_tracking_uri()}")
print(f"✓ Эксперимент: {mlflow.get_experiment_by_name('road-surface-kaggle').experiment_id}")

## 4. Обучение модели

In [ ]:
# Импорт модулей проекта
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Пример простой модели (замените на вашу архитектуру)
class AudioClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = torch.hub.load('pytorch/vision', 'resnet18', pretrained=True)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        return self.backbone(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioClassifier().to(device)
print(f"✓ Модель создана на {device}")

In [ ]:
# Конфигурация обучения
config = {
    "epochs": 10,
    "batch_size": 32,
    "learning_rate": 0.001,
    "num_classes": 5,
    "model_name": "resnet18",
    "experiment_name": "road-surface-kaggle",
}

print("Конфигурация обучения:")
for k, v in config.items():
    print(f"  {k}: {v}")

In [ ]:
# Запуск обучения с MLflow + WandB
from core.callbacks import DualLogger

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Инициализация логгера
with DualLogger(
    project="road-surface-classification",
    experiment_name=config["experiment_name"],
    wandb_config={"config": config},
    log_to_wandb=wandb_enabled,
    log_to_mlflow=True,
) as logger:
    
    # Логирование параметров
    logger.log_params(config)
    
    # MLflow run
    with mlflow.start_run(run_name="kaggle-training"):
        
        for epoch in range(config["epochs"]):
            model.train()
            
            # Здесь должен быть ваш training loop
            # Для примера — фиктивные метрики
            train_loss = 1.0 / (epoch + 1)
            val_loss = 1.2 / (epoch + 1)
            val_acc = 0.5 + epoch * 0.05
            
            # Логирование метрик
            metrics = {
                "train/loss": train_loss,
                "val/loss": val_loss,
                "val/accuracy": val_acc,
            }
            logger.log_metrics(metrics, step=epoch)
            
            print(f"Epoch {epoch+1}/{config['epochs']}: "
                  f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        
        # Сохранение модели
        logger.log_model(model, artifact_name="model")
        print("\n✓ Обучение завершено, модель залогирована")

## 5. Очистка

In [ ]:
# Очистка временных файлов
import shutil

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
    print(f"✓ Временная директория {WORK_DIR} удалена")

## Ссылки

- [MLflow UI](YOUR_MLFLOW_SERVER_URL) — просмотр экспериментов
- [WandB Dashboard](https://wandb.ai/your-entity/road-surface-classification) — визуализация
- [DVC Docs](https://dvc.org/doc) — управление данными